[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

# Reading and Writing ONNX Models — Deep Dive

This notebook covers the bridge between the abstract Protobuf hierarchy and everyday engineering: round-tripping `.onnx` files, controlling when weight bytes enter memory, using external data for large checkpoints, serialization complexity, and memory-aware inspection patterns.

| # | Section | Description |
|---|---------|-------------|
| 1 | [The Serialization Pipeline](#1-the-serialization-pipeline) | End-to-end flow from Python objects to bytes |
| 2 | [Core I/O APIs](#2-core-io-apis) | load, save, load_from_string, and their parameters |
| 3 | [Serialization Internals](#3-serialization-internals) | How Protobuf encodes the message tree |
| 4 | [Serialization Complexity](#4-serialization-complexity) | Big-O analysis of read/write operations |
| 5 | [External Data Format](#5-external-data-format) | Storing weights outside the Protobuf message |
| 6 | [External Data Thresholds](#6-external-data-thresholds) | When and why to externalize tensors |
| 7 | [Memory-Efficient Loading](#7-memory-efficient-loading) | Skeleton loads, lazy hydration, streaming |
| 8 | [In-Memory Round-Trip](#8-in-memory-round-trip) | SerializeToString / ParseFromString patterns |
| 9 | [Model Surgery Patterns](#9-model-surgery-patterns) | Modify, patch, and re-save models |
| 10 | [Byte-Level Inspection](#10-byte-level-inspection) | Hex dumps and tag decoding |
| 11 | [Performance Benchmarks](#11-performance-benchmarks) | Measuring I/O throughput |
| 12 | [Key Takeaways](#12-key-takeaways) | Summary |

In [ ]:
# !pip install onnx numpy matplotlib --quiet

import os
import tempfile
import time
import numpy as np
import onnx
from onnx import TensorProto, checker, helper, numpy_helper

## 1. The Serialization Pipeline

ONNX model persistence follows a well-defined pipeline that translates between Python objects and bytes on disk.

### Write Path

```
┌──────────────┐    SerializeToString()    ┌──────────────┐    write()    ┌───────────┐
│  ModelProto   │─────────────────────────▶│  bytes object │────────────▶│ .onnx file │
│  (Python obj) │                          │  (in memory)  │             │  (on disk) │
└──────────────┘                           └──────────────┘             └───────────┘
```

### Read Path

```
┌───────────┐    read()     ┌──────────────┐    ParseFromString()    ┌──────────────┐
│ .onnx file │────────────▶│  bytes object │─────────────────────▶  │  ModelProto   │
│  (on disk) │             │  (in memory)  │                        │  (Python obj) │
└───────────┘              └──────────────┘                         └──────────────┘
```

### With External Data

```
┌───────────┐    read()     ┌─────────────────┐    ParseFromString()    ┌──────────────┐
│ .onnx file │────────────▶│  skeleton bytes  │─────────────────────▶  │  ModelProto   │
│ (metadata) │             │  (no weight data)│                        │  (no weights) │
└───────────┘              └─────────────────┘                         └──────────────┘
                                                                              │
┌─────────────┐    mmap/read    ┌───────────────┐   load_external_data()      │
│ weights.bin  │───────────────▶│  tensor bytes  │───────────────────────────▶ │
│ (on disk)    │                │  (hydrated)    │                        ┌────┴────────┐
└─────────────┘                 └───────────────┘                        │  ModelProto  │
                                                                         │  (complete)  │
                                                                         └─────────────┘
```

Under the hood, `onnx.save_model()` calls `SerializeToString()` and writes the result; `onnx.load_model()` reads bytes and calls `ParseFromString()`. The serialized message layout is a concatenation of field encodings:

$$\text{Message bytes} = \bigoplus_{i=1}^{N} (\text{tag}_i \;\|\; \text{value}_i)$$

where $\bigoplus$ denotes byte concatenation.

## 2. Core I/O APIs

### API Summary

| Function | Direction | Key Parameters |
|----------|-----------|----------------|
| `onnx.load_model(f)` | disk → Python | `load_external_data`, `format` |
| `onnx.save_model(model, f)` | Python → disk | `save_as_external_data`, `all_tensors_to_one_file`, `size_threshold`, `location` |
| `onnx.load_model_from_string(s)` | bytes → Python | — |
| `model.SerializeToString()` | Python → bytes | — |
| `model.ParseFromString(s)` | bytes → Python (in-place) | — |

### Aliases

- `onnx.load` = `onnx.load_model`
- `onnx.save` = `onnx.save_model`

### Load Parameters

| Parameter | Default | Purpose |
|-----------|---------|--------|
| `f` | *(required)* | Path string or file-like object |
| `format` | `None` | Force format (`protobuf` or `textproto`) |
| `load_external_data` | `True` | When `False`, external tensor data is not loaded |

### Save Parameters for External Data

| Parameter | Default | Purpose |
|-----------|---------|--------|
| `save_as_external_data` | `False` | Enable external data storage |
| `all_tensors_to_one_file` | `True` | Single vs per-tensor files |
| `location` | `None` | External data filename |
| `size_threshold` | `1024` | Min bytes to externalize a tensor |
| `convert_attribute` | `False` | Also externalize attribute tensors |

In [ ]:
def build_toy_model(n_features=3, n_outputs=2):
    """Build a toy MatMul model with explicit weights."""
    x = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, n_features])
    y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, n_outputs])
    w_arr = np.random.randn(n_features, n_outputs).astype(np.float32)
    w_init = numpy_helper.from_array(w_arr, name="W")
    node = helper.make_node("MatMul", ["X", "W"], ["Y"])
    graph = helper.make_graph([node], "tiny_matmul", [x], [y], initializer=[w_init])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model.producer_name = "io_tutorial"
    checker.check_model(model, full_check=True)
    return model

model = build_toy_model()
print(f"Built model with {len(model.graph.initializer)} initializer(s)")
print(f"Serialized size: {len(model.SerializeToString())} bytes")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "model.onnx")

    onnx.save_model(model, path)
    file_size = os.path.getsize(path)
    print(f"Saved to disk: {file_size} bytes")

    loaded = onnx.load_model(path)
    checker.check_model(loaded, full_check=True)
    print(f"Loaded graph: {loaded.graph.name}")
    print(f"  Nodes: {len(loaded.graph.node)}")
    print(f"  Op types: {[n.op_type for n in loaded.graph.node]}")

    assert model.SerializeToString() == loaded.SerializeToString()
    print("\nByte-for-byte round-trip confirmed!")

## 3. Serialization Internals

When Protobuf serializes a `ModelProto`, it recursively encodes each field as a (tag, value) pair. The encoding depends on the field's wire type:

### Wire Type Encoding Summary

```
Wire Type 0 (Varint):     [tag_varint] [value_varint]
Wire Type 1 (64-bit):     [tag_varint] [8 bytes LE]
Wire Type 2 (Len-delim):  [tag_varint] [length_varint] [data bytes]
Wire Type 5 (32-bit):     [tag_varint] [4 bytes LE]
```

### Nested Message Encoding

Sub-messages (like `GraphProto` inside `ModelProto`) are encoded as length-delimited fields. The encoder must compute the sub-message's size before writing it:

$$\text{SubMessage} = \text{tag}_{\text{parent}} \;\|\; \text{varint}(|M_{\text{child}}|) \;\|\; M_{\text{child}}$$

This means serialization requires **two passes** for nested messages:
1. **Size computation pass**: recursively compute sizes bottom-up
2. **Write pass**: emit bytes top-down

### TensorProto Raw Data

For `raw_data` fields in `TensorProto`, the encoding is particularly efficient — the tensor bytes are written directly as a length-delimited blob:

$$\text{TensorData} = \text{tag}_{\text{raw\_data}} \;\|\; \text{varint}(N \times s) \;\|\; \underbrace{b_1 b_2 \cdots b_{Ns}}_{\text{little-endian tensor bytes}}$$

where $N = \prod d_i$ is the number of elements and $s$ is the element size in bytes.

In [ ]:
raw = model.SerializeToString()

print("First 100 bytes — hex dump with tag decoding:")
print(f"{'Offset':<8} {'Hex':<48} {'ASCII':<18} {'Tag decode'}")
print("-" * 100)

for offset in range(0, min(len(raw), 100), 16):
    chunk = raw[offset:offset+16]
    hex_str = ' '.join(f'{b:02x}' for b in chunk)
    ascii_str = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)

    tag_info = ""
    if offset == 0:
        tag_val = chunk[0]
        field_num = tag_val >> 3
        wire_type = tag_val & 0x07
        wt_names = {0: 'varint', 1: '64-bit', 2: 'len-delim', 5: '32-bit'}
        tag_info = f"field={field_num} wire={wt_names.get(wire_type, '?')}"

    print(f"{offset:06x}  {hex_str:<48} {ascii_str:<18} {tag_info}")

print(f"\nTotal serialized: {len(raw)} bytes")

## 4. Serialization Complexity

### Time Complexity

Both serialization and deserialization are **linear** in the message size:

$$T_{\text{serialize}} = O(S) \quad \text{where } S = \text{total serialized bytes}$$

$$T_{\text{deserialize}} = O(S)$$

For an ONNX model with $K$ initializer tensors, each with $N_k$ elements of size $s_k$ bytes:

$$S = S_{\text{meta}} + \sum_{k=1}^{K} (c_k + N_k \cdot s_k)$$

where $c_k$ is the per-tensor Protobuf overhead (tag + length prefix + name + dims, typically 20–100 bytes), and $S_{\text{meta}}$ is the graph structure overhead.

Since weight data dominates for real models:

$$S \approx \sum_{k=1}^{K} N_k \cdot s_k = S_{\text{weights}}$$

### Space Complexity

During serialization, Protobuf needs temporary buffers for size computation:

$$\text{RAM}_{\text{serialize}} = S + O(D) \quad \text{(depth-proportional stack for size computation)}$$

where $D$ is the nesting depth of the message tree (typically 4–6 for ONNX).

During deserialization, the parsed `ModelProto` uses more memory than the serialized form due to Python object overhead:

$$\text{RAM}_{\text{parsed}} \approx S_{\text{weights}} + N_{\text{objects}} \cdot c_{\text{py\_overhead}}$$

where $c_{\text{py\_overhead}} \approx 100$–$200$ bytes per Python object.

### Disk I/O Complexity

File I/O adds the disk latency component:

$$T_{\text{load}} = T_{\text{seek}} + \frac{S}{\text{bandwidth}} + T_{\text{parse}}$$

For SSDs with ~3 GB/s bandwidth, a 100 MB model loads in:

$$T \approx 0 + \frac{10^8}{3 \times 10^9} + O(10^8) \approx 33\text{ms (I/O)} + 50\text{ms (parse)}$$

In [ ]:
import matplotlib.pyplot as plt

def build_model_with_weights(n_params):
    x = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, n_params])
    y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, n_params])
    w = numpy_helper.from_array(np.random.randn(n_params).astype(np.float32), name="W")
    node = helper.make_node("Add", ["X", "W"], ["Y"])
    graph = helper.make_graph([node], "bench", [x], [y], initializer=[w])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

param_sizes = [1000, 5000, 10000, 50000, 100000, 500000]
ser_times, deser_times, sizes = [], [], []

print(f"{'Params':>10} | {'Size (KB)':>10} | {'Ser (ms)':>10} | {'Deser (ms)':>10} | {'Throughput':>12}")
print("-" * 62)

for n in param_sizes:
    m = build_model_with_weights(n)
    reps = 10

    t0 = time.perf_counter()
    for _ in range(reps):
        raw = m.SerializeToString()
    ser_ms = (time.perf_counter() - t0) / reps * 1000

    t0 = time.perf_counter()
    for _ in range(reps):
        p = onnx.ModelProto()
        p.ParseFromString(raw)
    deser_ms = (time.perf_counter() - t0) / reps * 1000

    size_kb = len(raw) / 1024
    throughput = (len(raw) / 1e6) / (ser_ms / 1000) if ser_ms > 0 else 0

    ser_times.append(ser_ms)
    deser_times.append(deser_ms)
    sizes.append(len(raw))

    print(f"{n:>10,} | {size_kb:>9.1f} | {ser_ms:>9.3f} | {deser_ms:>9.3f} | {throughput:>9.1f} MB/s")

print("\nBoth operations scale linearly with data size (O(n)).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(param_sizes, ser_times, 'o-', label='Serialize', linewidth=2, color='#2196F3')
ax.plot(param_sizes, deser_times, 's-', label='Deserialize', linewidth=2, color='#F44336')
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Time (ms)')
ax.set_title('Serialization Time vs Model Size', fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
throughputs = [(s / 1e6) / (t / 1000) if t > 0 else 0
               for s, t in zip(sizes, ser_times)]
ax.bar(range(len(param_sizes)), throughputs, color='#4CAF50', alpha=0.8)
ax.set_xlabel('Model Size Index')
ax.set_ylabel('Throughput (MB/s)')
ax.set_title('Serialization Throughput', fontweight='bold')
ax.set_xticks(range(len(param_sizes)))
ax.set_xticklabels([f'{n//1000}K' for n in param_sizes], rotation=30)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. External Data Format

When ONNX stores weights **outside** the primary Protobuf message, each `TensorProto` gets special fields:

- `data_location` set to `EXTERNAL` (value 1)
- `external_data` containing key-value pairs: `location`, `offset`, `length`, `checksum`

### Why External Data?

Protobuf has a practical size limit of approximately $2^{31} - 1$ bytes (~2 GB) for a single serialized message. Modern transformer models easily exceed this:

| Model | Parameters | Float32 Size | Needs External? |
|-------|-----------|-------------|----------------|
| ResNet-50 | 25.6M | 97.7 MB | No |
| BERT-Large | 340M | 1.3 GB | No (borderline) |
| GPT-2 XL | 1.5B | 5.7 GB | **Yes** |
| LLaMA-7B | 7B | 26.8 GB | **Yes** |

### External Data File Layout

```
model_directory/
├── model.onnx          ← Protobuf skeleton (graph structure, no weight data)
└── weights.bin          ← Raw tensor bytes concatenated
    ┌─────────────────────────────────────────────────────┐
    │ [Tensor A raw bytes] [Tensor B raw bytes] [...]     │
    │  offset=0            offset=|A|            ...      │
    │  length=|A|          length=|B|                     │
    └─────────────────────────────────────────────────────┘
```

Each `TensorProto` in the `.onnx` file references its data by:

$$\text{data}(T) = \text{weights.bin}[\text{offset}(T) \;:\; \text{offset}(T) + \text{length}(T)]$$

In [ ]:
model_ext = build_toy_model(n_features=100, n_outputs=50)

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "external.onnx")
    onnx.save_model(
        model_ext, path,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location="weights.bin",
        size_threshold=0,
    )

    print("Files created:")
    for fname in sorted(os.listdir(tmp)):
        fpath = os.path.join(tmp, fname)
        print(f"  {fname}: {os.path.getsize(fpath):,} bytes")

    skeleton = onnx.load_model(path, load_external_data=False)
    w0 = skeleton.graph.initializer[0]
    print(f"\nSkeleton load (external_data=False):")
    print(f"  data_location: {w0.data_location} (1=EXTERNAL)")
    print(f"  raw_data length: {len(w0.raw_data)} bytes (empty!)")
    print(f"  External data entries:")
    for ed in w0.external_data:
        print(f"    {ed.key} = {ed.value}")

In [ ]:
from onnx.external_data_helper import load_external_data_for_model

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "external.onnx")
    onnx.save_model(
        model_ext, path,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location="weights.bin",
        size_threshold=0,
    )

    m = onnx.load_model(path, load_external_data=False)
    print(f"Before hydration: raw_data = {len(m.graph.initializer[0].raw_data)} bytes")

    load_external_data_for_model(m, tmp)
    print(f"After hydration:  raw_data = {len(m.graph.initializer[0].raw_data)} bytes")

    checker.check_model(m, full_check=True)
    print("Model valid after hydration!")

    arr = numpy_helper.to_array(m.graph.initializer[0])
    print(f"Weight shape: {arr.shape}, dtype: {arr.dtype}")

## 6. External Data Thresholds

The `size_threshold` parameter controls which tensors get externalized. The decision rule is:

$$\text{externalize}(T) = \begin{cases} \text{true} & \text{if } |T_{\text{raw}}| \geq \text{threshold} \\ \text{false} & \text{otherwise} \end{cases}$$

### Threshold Strategy

| Threshold | Effect | Use case |
|-----------|--------|----------|
| `0` | Externalize everything | Maximum skeleton compactness |
| `1024` (default) | Small constants stay inline | Good general default |
| `1048576` (1MB) | Only large weights go external | Keep most data inline |
| Very large | Nothing externalized | Force single-file output |

### Protobuf Size Limit

The hard constraint driving external data:

$$S_{\text{protobuf}} < 2^{31} - 1 = 2{,}147{,}483{,}647 \;\text{bytes} \approx 2 \;\text{GB}$$

If $S_{\text{weights}} > S_{\text{limit}} - S_{\text{structure}}$, external data is **mandatory**.

### Per-Tensor vs All-in-One

```
all_tensors_to_one_file=True:        all_tensors_to_one_file=False:

model_dir/                           model_dir/
├── model.onnx                       ├── model.onnx
└── weights.bin  ← all tensors       ├── conv1.weight  ← one per tensor
                                     ├── conv1.bias
                                     ├── fc.weight
                                     └── fc.bias
```

In [ ]:
large_model = build_model_with_weights(100000)

thresholds = [0, 1024, 10000, 100000, 1000000]
print(f"{'Threshold':>12} | {'Model file':>12} | {'External':>12} | {'Externalized?':>14}")
print("-" * 58)

for thresh in thresholds:
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "model.onnx")
        onnx.save_model(
            large_model, path,
            save_as_external_data=True,
            all_tensors_to_one_file=True,
            location="weights.bin",
            size_threshold=thresh,
        )
        model_size = os.path.getsize(path)
        ext_path = os.path.join(tmp, "weights.bin")
        ext_size = os.path.getsize(ext_path) if os.path.exists(ext_path) else 0
        ext_flag = "Yes" if ext_size > 0 else "No"
        print(f"{thresh:>12,} | {model_size:>10,} B | {ext_size:>10,} B | {ext_flag:>14}")

## 7. Memory-Efficient Loading

For large models, loading the entire weight set into memory may be unnecessary or impractical. Several strategies help manage memory:

### Strategy Matrix

| Goal | Technique | Memory cost |
|------|----------|------------|
| Fast metadata inspection | `load_external_data=False` | $O(S_{\text{structure}})$ |
| Architecture analysis | Skeleton load + iterate nodes | $O(S_{\text{structure}})$ |
| Modify and re-save | Skeleton → modify → save (no re-read of weights) | $O(S_{\text{structure}})$ |
| Full inference setup | Full load or runtime lazy load | $O(S_{\text{total}})$ |
| Streaming at runtime | External data + runtime mmap | $O(S_{\text{working\_set}})$ |

### Skeleton Load Pattern

```python
# Memory-efficient: only loads graph structure
skeleton = onnx.load_model("huge_model.onnx", load_external_data=False)

# Inspect architecture without loading GB of weights
for node in skeleton.graph.node:
    print(node.op_type, list(node.input), list(node.output))

# Modify metadata
skeleton.doc_string = "Updated documentation"

# Save back (external data files are untouched)
onnx.save_model(skeleton, "huge_model_v2.onnx")
```

### Two-Stage Load

```
Stage 1: Load skeleton          Stage 2: Hydrate on demand
┌────────────────────┐          ┌─────────────────────────┐
│ Graph structure    │          │ Read weights.bin         │
│ Node definitions   │  ─────▶  │ Fill raw_data fields    │
│ Shape info         │          │ Model now complete       │
│ (No weight data)   │          │                          │
└────────────────────┘          └─────────────────────────┘
   RAM: ~KB                        RAM: ~GB
```

In [ ]:
big_model = build_model_with_weights(500000)

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "big.onnx")
    onnx.save_model(
        big_model, path,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location="weights.bin",
        size_threshold=0,
    )

    t0 = time.perf_counter()
    skeleton = onnx.load_model(path, load_external_data=False)
    t_skeleton = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    full = onnx.load_model(path, load_external_data=True)
    t_full = (time.perf_counter() - t0) * 1000

    print(f"Skeleton load: {t_skeleton:.2f} ms")
    print(f"Full load:     {t_full:.2f} ms")
    print(f"Speedup:       {t_full/t_skeleton:.1f}x faster for skeleton")
    print(f"\nSkeleton has weight data: {len(skeleton.graph.initializer[0].raw_data) > 0}")
    print(f"Full has weight data:     {len(full.graph.initializer[0].raw_data) > 0}")

    print(f"\nGraph inspection (works on skeleton):")
    for node in skeleton.graph.node:
        print(f"  {node.op_type}: {list(node.input)} -> {list(node.output)}")

## 8. In-Memory Round-Trip

Sometimes you need to serialize/deserialize without touching disk — for network transfer, in-memory caching, or passing models between processes.

### SerializeToString / ParseFromString

```python
# Serialize to bytes (no file I/O)
raw_bytes = model.SerializeToString()

# Deserialize from bytes
restored = onnx.load_model_from_string(raw_bytes)
```

### Use Cases

| Scenario | Method |
|----------|--------|
| Pass model via gRPC | `SerializeToString()` → transmit → `ParseFromString()` |
| Clone a model | `clone.ParseFromString(original.SerializeToString())` |
| Cache in Redis/memcached | Store `SerializeToString()` bytes |
| Compute hash/checksum | `hashlib.sha256(model.SerializeToString())` |
| Unit testing | Build model → serialize → parse → compare |

In [ ]:
import hashlib

raw = model.SerializeToString()
print(f"Serialized: {len(raw)} bytes")

roundtrip = onnx.load_model_from_string(raw)
checker.check_model(roundtrip, full_check=True)
print("In-memory round-trip OK!")

raw2 = roundtrip.SerializeToString()
assert raw == raw2, "Bytes differ after round-trip!"
print("Byte identity confirmed.")

digest = hashlib.sha256(raw).hexdigest()
print(f"\nSHA-256: {digest}")

clone = onnx.ModelProto()
clone.CopyFrom(model)
clone.doc_string = "Modified clone"
assert clone.SerializeToString() != raw, "Clone should differ"
print(f"\nOriginal doc: {model.doc_string!r}")
print(f"Clone doc:    {clone.doc_string!r}")
print("CopyFrom creates a deep copy — original unmodified.")

## 9. Model Surgery Patterns

Common model modifications that leverage the load/save pipeline:

### Pattern 1: Update Metadata

```
Load skeleton → modify metadata → save
(weights stay on disk, never loaded into RAM)
```

### Pattern 2: Remove/Replace Nodes

```
Load full → modify graph.node list → re-validate → save
```

### Pattern 3: Opset Upgrade

```
Load → version_converter.convert_version() → save
```

### Pattern 4: Shape Inference

```
Load → shape_inference.infer_shapes() → save
(populates value_info for all intermediate tensors)
```

In [ ]:
m = build_toy_model()

print("=== Pattern 1: Metadata Surgery ===")
m.producer_name = "surgery_demo"
m.producer_version = "2.0"
m.doc_string = "Updated via model surgery"
entry = m.metadata_props.add()
entry.key = "surgery_date"
entry.value = "2024-01-01"
print(f"  Producer: {m.producer_name} v{m.producer_version}")
print(f"  Metadata: {[(p.key, p.value) for p in m.metadata_props]}")

print("\n=== Pattern 2: Shape Inference ===")
inferred = onnx.shape_inference.infer_shapes(m)
print(f"  value_info before: {len(m.graph.value_info)}")
print(f"  value_info after:  {len(inferred.graph.value_info)}")

print("\n=== Pattern 3: Opset Conversion ===")
print(f"  Original opset: {m.opset_import[0].version}")
try:
    from onnx import version_converter
    converted = version_converter.convert_version(m, 15)
    print(f"  Converted opset: {converted.opset_import[0].version}")
    checker.check_model(converted)
    print("  Validation passed!")
except Exception as e:
    print(f"  Conversion: {e}")

## 10. Byte-Level Inspection

For debugging serialization issues or understanding the wire format, byte-level inspection is invaluable.

### Decoding the Wire Format

Each field on the wire starts with a tag varint:

$$\text{tag} = (\text{field\_number} \ll 3) \;|\; \text{wire\_type}$$

The decoder reads the tag, determines the wire type, and reads the appropriate number of bytes:

```
Read tag varint
  │
  ├── wire_type = tag & 0x07
  ├── field_number = tag >> 3
  │
  ├── if wire_type == 0: read varint value
  ├── if wire_type == 1: read 8 bytes (fixed64/double)
  ├── if wire_type == 2: read length varint, then read that many bytes
  └── if wire_type == 5: read 4 bytes (fixed32/float)
```

In [ ]:
def decode_varint(data, pos):
    """Decode varint at position, return (value, new_pos)."""
    value = 0
    shift = 0
    while pos < len(data):
        byte = data[pos]
        value |= (byte & 0x7F) << shift
        pos += 1
        if not (byte & 0x80):
            break
        shift += 7
    return value, pos

def dump_protobuf_fields(data, max_fields=15):
    """Decode and print top-level Protobuf fields."""
    wire_names = {0: 'Varint', 1: '64-bit', 2: 'Len-delim', 5: '32-bit'}
    pos = 0
    count = 0
    while pos < len(data) and count < max_fields:
        start = pos
        tag, pos = decode_varint(data, pos)
        wire_type = tag & 0x07
        field_num = tag >> 3
        wt_name = wire_names.get(wire_type, '???')

        if wire_type == 0:
            val, pos = decode_varint(data, pos)
            print(f"  @{start:4d}  field={field_num:3d}  {wt_name:<12s}  value={val}")
        elif wire_type == 2:
            length, pos = decode_varint(data, pos)
            preview = data[pos:pos+min(length, 30)]
            pos += length
            try:
                text = preview.decode('utf-8', errors='strict')
                if all(32 <= ord(c) < 127 for c in text):
                    print(f"  @{start:4d}  field={field_num:3d}  {wt_name:<12s}  len={length}  text={text!r}")
                else:
                    print(f"  @{start:4d}  field={field_num:3d}  {wt_name:<12s}  len={length}  hex={preview[:16].hex()}...")
            except UnicodeDecodeError:
                print(f"  @{start:4d}  field={field_num:3d}  {wt_name:<12s}  len={length}  hex={preview[:16].hex()}...")
        elif wire_type == 5:
            val = data[pos:pos+4]
            pos += 4
            print(f"  @{start:4d}  field={field_num:3d}  {wt_name:<12s}  hex={val.hex()}")
        elif wire_type == 1:
            val = data[pos:pos+8]
            pos += 8
            print(f"  @{start:4d}  field={field_num:3d}  {wt_name:<12s}  hex={val.hex()}")
        else:
            print(f"  @{start:4d}  field={field_num:3d}  wire_type={wire_type} (unknown)")
            break
        count += 1

raw = model.SerializeToString()
print(f"ModelProto wire format ({len(raw)} bytes):")
dump_protobuf_fields(raw)

## 11. Performance Benchmarks

### End-to-End I/O Timing

Complete save and load cycle including disk I/O:

![Linear Regression Model Graph](assets/dot_linreg2.png)

In [ ]:
io_sizes = [1000, 10000, 50000, 200000, 500000]
save_times, load_times = [], []

print(f"{'Params':>10} | {'Save (ms)':>10} | {'Load (ms)':>10} | {'Size (KB)':>10}")
print("-" * 50)

for n in io_sizes:
    m = build_model_with_weights(n)

    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "bench.onnx")

        t0 = time.perf_counter()
        for _ in range(5):
            onnx.save_model(m, path)
        save_ms = (time.perf_counter() - t0) / 5 * 1000

        t0 = time.perf_counter()
        for _ in range(5):
            _ = onnx.load_model(path)
        load_ms = (time.perf_counter() - t0) / 5 * 1000

        size_kb = os.path.getsize(path) / 1024

    save_times.append(save_ms)
    load_times.append(load_ms)
    print(f"{n:>10,} | {save_ms:>9.2f} | {load_ms:>9.2f} | {size_kb:>9.1f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(io_sizes, save_times, 'o-', label='Save (serialize + write)', linewidth=2, color='#2196F3')
ax.plot(io_sizes, load_times, 's-', label='Load (read + parse)', linewidth=2, color='#F44336')
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Time (ms)')
ax.set_title('ONNX I/O Performance (Disk Round-Trip)', fontweight='bold')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Key Takeaways

1. **`onnx.load_model` / `onnx.save_model`** are the primary I/O APIs, wrapping Protobuf's `ParseFromString` / `SerializeToString` with file I/O.

2. **Serialization is $O(n)$** in total message size, with two internal passes (size computation + write). Deserialization is also $O(n)$ in a single pass.

3. **External data** keeps `.onnx` files compact when model weights exceed the ~2 GB Protobuf limit. The threshold formula:

$$\text{externalize}(T) = \begin{cases} \text{true} & \text{if } |T_{\text{raw}}| \geq \text{threshold} \\ \text{false} & \text{otherwise} \end{cases}$$

4. **`load_external_data=False`** enables memory-efficient skeleton loading — inspect architecture and modify metadata without loading GB of weights.

5. **In-memory round-trips** via `SerializeToString()` / `load_model_from_string()` are useful for cloning, hashing, network transfer, and testing.

6. **Model surgery patterns** (metadata update, node modification, opset conversion, shape inference) all follow the load → modify → validate → save pipeline.

7. Always treat `.onnx` + external data blobs as a **bundle** — keep them together in the same directory. Moving one without the other breaks the model.

8. **Byte-level inspection** using tag decoding reveals the Protobuf wire format structure: $\text{tag} = (\text{field\_number} \ll 3) | \text{wire\_type}$.